# Protein interface analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davismerighi-max/interface-analysis/blob/main/notebooks/interface_analysis_colab.ipynb)

Give it a refined structure (`.pdb` / `.cif`, or a PDB entry code) and it measures every
interface: which residues are buried and by how much, the salt bridges, hydrogen bonds,
hydrophobic contacts, disulfides, metal bridges and stacking that hold the subunits
together, how well the surfaces fit, what bound cofactors touch, and which parts of the
model could be stable assemblies on their own. Then it draws the figures and hands you
everything as a zip.

**No programming needed.** Run the cells from top to bottom (`Shift`+`Enter`, or ▶ on each
cell). Where a cell has boxes, fill them in first. Symmetry, chain count and the number of
different proteins are measured from your model, never declared.

Code and documentation: https://github.com/davismerighi-max/interface-analysis


## 1 · Set up

Installs the libraries and downloads the code. About a minute; once per session.


In [ ]:
#@title Install and fetch the code { display-mode: "form" }
import os, sys, subprocess, shutil, pathlib
print("installing libraries ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "biopython", "pydssp", "scikit-image"], check=True)
REPO = "/content/interface-analysis"
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "-q",
                "https://github.com/davismerighi-max/interface-analysis.git", REPO], check=True)
WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)
for f in pathlib.Path(REPO).glob("*.py"):
    shutil.copy(f, WORK)
os.chdir(WORK)
print("ready — working directory is", os.getcwd())

## 2 · Give it a structure

Type a four-character PDB entry code, or leave the box empty and upload your own file.


In [ ]:
#@title Load the model { display-mode: "form" }
pdb_entry_code = "2TSC"  #@param {type:"string"}
#@markdown Leave the box empty to upload your own `.pdb` / `.cif` file instead.
import urllib.request
code_ = pdb_entry_code.strip().lower()
if code_:
    MODEL = code_ + ".pdb"
    print("downloading", MODEL)
    urllib.request.urlretrieve("https://files.rcsb.org/download/" + MODEL, MODEL)
else:
    from google.colab import files
    MODEL = list(files.upload())[0]
n = sum(1 for l in open(MODEL, errors="ignore") if l.startswith(("ATOM", "HETATM")))
print("model:", MODEL, "-", n, "atom records")

## 3 · Name and colour your proteins (optional)

One row per protein: a **name**, a **colour** as `#RRGGBB`, and its **sequence**. Proteins are
recognised in the model **by sequence**, so the order does not matter and the sequence may be
the full-length one even if your model is missing loops or termini. Leave everything empty to
use the defaults (`entity_1`, `entity_2`, … in the standard palette). Anything that does not
match the model runs with defaults, and the run report tells you which and why.

Keep names short — they label every axis.


In [ ]:
#@title Proteins { display-mode: "form" }
protein_1_name = ""  #@param {type:"string"}
protein_1_colour = ""  #@param {type:"string"}
protein_1_sequence = ""  #@param {type:"string"}
protein_2_name = ""  #@param {type:"string"}
protein_2_colour = ""  #@param {type:"string"}
protein_2_sequence = ""  #@param {type:"string"}
protein_3_name = ""  #@param {type:"string"}
protein_3_colour = ""  #@param {type:"string"}
protein_3_sequence = ""  #@param {type:"string"}
protein_4_name = ""  #@param {type:"string"}
protein_4_colour = ""  #@param {type:"string"}
protein_4_sequence = ""  #@param {type:"string"}
#@markdown Or upload a FASTA file instead (headers like `>YfdQ_#7FA8D4`):
upload_fasta_instead = False  #@param {type:"boolean"}

PROTEINS = None
if upload_fasta_instead:
    from google.colab import files
    PROTEINS = list(files.upload())[0]
else:
    rows = []
    for i in range(1, 5):
        name = globals()[f"protein_{i}_name"].strip().replace(" ", "_")
        col = globals()[f"protein_{i}_colour"].strip()
        seq = "".join(globals()[f"protein_{i}_sequence"].split())
        if name and seq:
            hdr = name + ("_" + col if col else "")
            rows.append(">" + hdr + "\n" + seq + "\n")
        elif name or seq:
            print(f"protein {i}: needs both a name and a sequence; skipped")
    if rows:
        PROTEINS = "proteins.fasta"
        open(PROTEINS, "w").write("".join(rows))
print("protein details:", PROTEINS or "none given, defaults will be used")

## 4 · Measure the interfaces

The main calculation: symmetry, contacts, buried surface atom by atom, and the chemistry of
every contact. `resolution` is that of your map or diffraction data (0 if unknown).
*Shape complementarity* is the slow option — about a minute for a dimer, ten minutes or more
for 30 chains. Switch it off for a first look.


In [ ]:
#@title Run step 1 { display-mode: "form" }
resolution = 1.97  #@param {type:"number"}
shape_complementarity = True  #@param {type:"boolean"}
cmd = f"python 01_interface_analysis.py {MODEL}"
if resolution and resolution > 0:
    cmd += f" --resolution {resolution:g}"
if shape_complementarity:
    cmd += " --shape --shape-workers 2"
if PROTEINS:
    cmd += f" --proteins {PROTEINS}"
print(cmd, "\n")
!{cmd}

## 5 · Tables and statistics

No settings: it reads them back from step 1.


In [ ]:
#@title Run step 2 and show the statistics { display-mode: "form" }
!python 02_derive_tables.py {MODEL}
print("\n" + "=" * 78 + "\n")
print(open("stats_summary.txt", encoding="utf-8").read())

## 6 · Figures

Besides the 2D figures, step 3 draws two quick 3D pictures of the whole model in the same
licorice style as the assembly renders: one coloured by the protein that buries each residue
(the *Interface budget*), one by the strongest interaction each residue makes (*Directional
interactions*). They are for a fast look; for rotating and close-ups use the ChimeraX files in
the download. Interfaces are mostly inside an assembly, so try `view_elevation = 0` for a
side view.


In [ ]:
#@title Run step 3 { display-mode: "form" }
residues_per_line = 350  #@param {type:"integer"}
draw_3D_renders = True  #@param {type:"boolean"}
view_elevation = 62  #@param {type:"number"}
view_azimuth = 20  #@param {type:"number"}
_render = (f"--render-view {view_elevation},{view_azimuth}" if draw_3D_renders
           else "--no-render")
!python 03_make_figures.py {MODEL} --per-line {residues_per_line} {_render}


In [ ]:
#@title Show the figures here { display-mode: "form" }
import glob, base64, os
from IPython.display import Image, display, Markdown, HTML
renders = [f for f in ("fig_render_budget.png", "fig_render_interactions.png")
           if os.path.exists(f)]
figs = [f for f in sorted(glob.glob("fig*.png")) if f not in renders]
if not figs:
    print("no figures found - see the run report below")
if renders:
    display(Markdown("### 3D renders of the whole model"))
    imgs = "".join(
        f'<img src="data:image/png;base64,{base64.b64encode(open(f, "rb").read()).decode()}" '
        f'style="width:460px;margin-right:12px">' for f in renders)
    display(HTML(f'<div style="display:flex;flex-wrap:wrap">{imgs}</div>'))
for f in figs:
    display(Markdown("### " + f))
    display(Image(filename=f, width=950))


## 7 · Which parts could be stable assemblies?

Takes the model apart weakest interface first and classifies every piece that appears, with
the reason. **kept** = share of a piece's buried area that stays inside it:
≥ 67 % strong candidate · ≥ 50 % candidate · < 50 % not supported by contacts alone ·
*covalent unit* = chains joined by disulfides. These cut-offs are a stated heuristic, not a
published standard; disulfides are reported separately, never counted as area. Treat the
table as evidence to test (SEC-MALS, mass photometry, non-reducing gels), not as a verdict.


In [ ]:
#@title Run step 5 and show the candidates { display-mode: "form" }
draw_assemblies = True  #@param {type:"boolean"}
!python 05_assemblies.py {MODEL} {"--render" if draw_assemblies else ""} > /dev/null
import pandas as pd, os
from IPython.display import display, Image
pd.set_option("display.max_colwidth", 200)
if os.path.exists("assembly_candidates.csv"):
    t = pd.read_csv("assembly_candidates.csv")
    display(t[["rank", "class", "kept_pct", "n_chains", "composition",
               "disulfides_inside", "disulfides_across_boundary", "reason"]])
if draw_assemblies and os.path.exists("fig_assemblies.png"):
    display(Image(filename="fig_assemblies.png", width=950))

## 8 · Run report

What worked, what ran with defaults, and what failed — check this before trusting any figure.


In [ ]:
#@title Show the run report { display-mode: "form" }
print(open("run_report.txt", encoding="utf-8").read())
if os.path.exists("run_errors.log"):
    print("\nDetails of any failure are in run_errors.log (included in the zip).")

## 9 · Take everything with you

Tables are plain CSV; figures are PNG (600 dpi) and vector PDF; the ChimeraX scripts open your model in flat, silhouetted licorice.


In [ ]:
#@title Download a zip of all results { display-mode: "form" }
import shutil
base = os.path.splitext(os.path.basename(MODEL))[0] + "_interface_analysis"
shutil.make_archive("/content/" + base, "zip", "/content/work")
print("packed", base + ".zip")
try:
    from google.colab import files
    files.download("/content/" + base + ".zip")
except Exception:
    print("the file is at /content/" + base + ".zip")

---
### Reading three of the numbers

**Buried area** is computed pairwise in isolation and reported halved (the interface area,
not the sum of both footprints): PISA's convention, so directly comparable to it.

**Shape complementarity** reads about **0.10 below** the original `sc` program (trypsin–BPTI
0.647; obligate histone heterodimers 0.598–0.620; published ≈ 0.70), so the class names on the
figure are pessimistic. Use S_c to rank interfaces within one run.

**ΔG** is the Eisenberg–McLachlan solvation term (20 amino acids only). Negative = hydrophobic
burial pays; positive = the interface is held by hydrogen bonds and salt bridges instead. It is
not a binding free energy.

### If something goes wrong
Look at the run report (section 8): each item says *ok*, *DEFAULT*, *WARNING*, *SKIPPED* or
*FAILED*, with the reason. A failure in one figure never stops the others. Very large
assemblies may exceed the free Colab machine's memory — run them locally or on a cluster; the
code is identical.
